In [1]:
import torch
import transformers
import datasets
import accelerate
import sentencepiece

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU available:", torch.cuda.is_available())

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", device)

PyTorch: 2.13.0+cpu
Transformers: 5.16.1
GPU available: False
Training device: cpu


In [ ]:
import torch
import transformers
import datasets
import accelerate
import sentencepiece

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU available:", torch.cuda.is_available())

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", device)

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(
    r"C:\Users\delll\Desktop\SmartRoute"
)

# Load the complete raw dataset
raw_path = (
    PROJECT_ROOT
    / "data/raw/multilingual_grievances.csv"
)

df = pd.read_csv(raw_path)

# Restore text features if needed
df["text_length"] = df["text"].astype(str).str.len()
df["word_count"] = df["text"].astype(str).str.split().str.len()

# Recreate the reliable group-aware split
rng = np.random.default_rng(42)
group_to_split = {}

for category in sorted(df["category"].unique()):
    for language in sorted(df["language"].unique()):

        subset = df[
            (df["category"] == category)
            & (df["language"] == language)
        ]

        groups = (
            subset["template_group"]
            .drop_duplicates()
            .astype(str)
            .to_numpy(dtype=object, copy=True)
        )

        groups = rng.permutation(groups)

        for group in groups[:7]:
            group_to_split[group] = "train"

        for group in groups[7:8]:
            group_to_split[group] = "validation"

        for group in groups[8:]:
            group_to_split[group] = "test"

df["split"] = (
    df["template_group"]
    .astype(str)
    .map(group_to_split)
)

# Validate before saving
assert df["split"].isnull().sum() == 0
assert df.groupby("template_group")["split"].nunique().max() == 1

expected_counts = {
    "train": 2520,
    "validation": 360,
    "test": 720
}

assert df["split"].value_counts().to_dict() == expected_counts

# Save the recovered split dataset
split_path = (
    PROJECT_ROOT
    / "data/processed/multilingual_grievances_split.csv"
)

split_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(
    split_path,
    index=False,
    encoding="utf-8-sig"
)

print("✅ Split dataset recovered and saved!")
print("File exists:", split_path.exists())
print("Shape:", df.shape)
print(df["split"].value_counts())

✅ Split dataset recovered and saved!
File exists: True
Shape: (3600, 10)
split
train         2520
test           720
validation     360
Name: count, dtype: int64


In [6]:
train_df = df[df["split"] == "train"].copy()
validation_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()

label_names = sorted(df["category"].unique())

label2id = {
    label: index
    for index, label in enumerate(label_names)
}

id2label = {
    index: label
    for label, index in label2id.items()
}

for current_df in [train_df, validation_df, test_df]:
    current_df["label"] = current_df["category"].map(label2id)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)
print("\nLabels:", label2id)

Train: (2520, 11)
Validation: (360, 11)
Test: (720, 11)

Labels: {'Account Issue': 0, 'Cancellation Issue': 1, 'Delivery Issue': 2, 'Fraud/Security': 3, 'Payment Issue': 4, 'Product Issue': 5, 'Refund Issue': 6, 'Return/Replacement': 7}


In [7]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df[["text", "label"]],
    preserve_index=False
)

validation_dataset = Dataset.from_pandas(
    validation_df[["text", "label"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["text", "label"]],
    preserve_index=False
)

print(train_dataset)
print(validation_dataset)
print(test_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 2520
})
Dataset({
    features: ['text', 'label'],
    num_rows: 360
})
Dataset({
    features: ['text', 'label'],
    num_rows: 720
})


In [8]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert/distilbert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=64
    )

tokenized_train = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

tokenized_validation = validation_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

tokenized_test = test_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

print("✅ All datasets tokenized successfully!")
print(tokenized_train)

Map:   0%|          | 0/2520 [00:00<?, ? examples/s]

Map:   0%|          | 0/360 [00:00<?, ? examples/s]

Map:   0%|          | 0/720 [00:00<?, ? examples/s]

✅ All datasets tokenized successfully!
Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2520
})


In [9]:
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt"
)

print("Multilingual DistilBERT loaded successfully!")

model.safetensors: reconstructing file:   0%|          |  0.00B /  542MB            

model.safetensors: downloading bytes:           |  0.00B            

C:\Users\delll\anaconda3\envs\smartroute\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\delll\.cache\huggingface\hub\models--distilbert--distilbert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Multilingual DistilBERT loaded successfully!


In [10]:
# Freeze the complete pretrained DistilBERT base
for parameter in model.distilbert.parameters():
    parameter.requires_grad = False

# Unfreeze the final two transformer layers
for layer in model.distilbert.transformer.layer[-2:]:
    for parameter in layer.parameters():
        parameter.requires_grad = True

# The pre-classifier and classifier remain trainable
trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(f"Total parameters: {total_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,}")
print(
    f"Trainable percentage: "
    f"{100 * trainable_parameters / total_parameters:.2f}%"
)

Total parameters: 135,330,824
Trainable parameters: 14,772,488
Trainable percentage: 10.92%


In [11]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    tokenized_train,
    batch_size=8,
    shuffle=True,
    collate_fn=data_collator,
    num_workers=0
)

validation_loader = DataLoader(
    tokenized_validation,
    batch_size=8,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=0
)

test_loader = DataLoader(
    tokenized_test,
    batch_size=8,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=0
)

device = torch.device("cpu")
model.to(device)

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Testing batches:", len(test_loader))
print("Device:", device)

Training batches: 315
Validation batches: 45
Testing batches: 90
Device: cpu


In [12]:
import time

import torch
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)
from transformers import get_linear_schedule_with_warmup

EPOCHS = 1
LEARNING_RATE = 2e-5

trainable_parameters = filter(
    lambda parameter: parameter.requires_grad,
    model.parameters()
)

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LEARNING_RATE,
    weight_decay=0.01
)

total_training_steps = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

training_history = []

print("Starting CPU training...")
print("Total batches:", len(train_loader))
print("Do not close Jupyter while training.\n")

training_start = time.perf_counter()

for epoch in range(EPOCHS):
    model.train()

    total_train_loss = 0
    epoch_start = time.perf_counter()

    for batch_number, batch in enumerate(train_loader, start=1):
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer.zero_grad()

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()

        if batch_number % 25 == 0:
            elapsed = time.perf_counter() - epoch_start
            average_batch_time = elapsed / batch_number
            remaining_batches = len(train_loader) - batch_number
            estimated_remaining = (
                average_batch_time * remaining_batches / 60
            )

            print(
                f"Epoch {epoch + 1} | "
                f"Batch {batch_number}/{len(train_loader)} | "
                f"Loss: {loss.item():.4f} | "
                f"Estimated remaining: "
                f"{estimated_remaining:.1f} minutes"
            )

    average_train_loss = total_train_loss / len(train_loader)

    # Validation
    model.eval()

    validation_labels = []
    validation_predictions = []
    total_validation_loss = 0

    with torch.no_grad():
        for batch in validation_loader:
            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            outputs = model(**batch)

            total_validation_loss += outputs.loss.item()

            predictions = torch.argmax(
                outputs.logits,
                dim=-1
            )

            validation_labels.extend(
                batch["labels"].cpu().numpy()
            )

            validation_predictions.extend(
                predictions.cpu().numpy()
            )

    validation_accuracy = accuracy_score(
        validation_labels,
        validation_predictions
    )

    val_precision, val_recall, val_f1, _ = (
        precision_recall_fscore_support(
            validation_labels,
            validation_predictions,
            average="weighted",
            zero_division=0
        )
    )

    average_validation_loss = (
        total_validation_loss / len(validation_loader)
    )

    epoch_minutes = (
        time.perf_counter() - epoch_start
    ) / 60

    training_history.append({
        "epoch": epoch + 1,
        "training_loss": average_train_loss,
        "validation_loss": average_validation_loss,
        "validation_accuracy": validation_accuracy,
        "validation_precision": val_precision,
        "validation_recall": val_recall,
        "validation_f1": val_f1,
        "epoch_minutes": epoch_minutes
    })

    print("\nEPOCH RESULT")
    print("-" * 40)
    print(f"Training loss:      {average_train_loss:.4f}")
    print(f"Validation loss:    {average_validation_loss:.4f}")
    print(f"Validation accuracy:{validation_accuracy:.4f}")
    print(f"Validation F1:      {val_f1:.4f}")
    print(f"Epoch time:         {epoch_minutes:.2f} minutes")

total_minutes = (
    time.perf_counter() - training_start
) / 60

print(f"\n✅ Training finished in {total_minutes:.2f} minutes")

Starting CPU training...
Total batches: 315
Do not close Jupyter while training.

Epoch 1 | Batch 25/315 | Loss: 2.0317 | Estimated remaining: 2.5 minutes
Epoch 1 | Batch 50/315 | Loss: 2.0462 | Estimated remaining: 2.1 minutes
Epoch 1 | Batch 75/315 | Loss: 2.0707 | Estimated remaining: 1.8 minutes
Epoch 1 | Batch 100/315 | Loss: 2.0422 | Estimated remaining: 1.6 minutes
Epoch 1 | Batch 125/315 | Loss: 2.0477 | Estimated remaining: 1.4 minutes
Epoch 1 | Batch 150/315 | Loss: 2.0339 | Estimated remaining: 1.2 minutes
Epoch 1 | Batch 175/315 | Loss: 2.0138 | Estimated remaining: 1.0 minutes
Epoch 1 | Batch 200/315 | Loss: 1.8098 | Estimated remaining: 0.9 minutes
Epoch 1 | Batch 225/315 | Loss: 1.9040 | Estimated remaining: 0.7 minutes
Epoch 1 | Batch 250/315 | Loss: 1.7390 | Estimated remaining: 0.5 minutes
Epoch 1 | Batch 275/315 | Loss: 1.6826 | Estimated remaining: 0.3 minutes
Epoch 1 | Batch 300/315 | Loss: 1.8275 | Estimated remaining: 0.1 minutes

EPOCH RESULT
-------------------

In [13]:
# Unfreeze the final four transformer layers
for layer in model.distilbert.transformer.layer[-4:]:
    for parameter in layer.parameters():
        parameter.requires_grad = True

trainable_count = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

total_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(
    f"Trainable parameters: {trainable_count:,} "
    f"({100 * trainable_count / total_count:.2f}%)"
)

Trainable parameters: 28,948,232 (21.39%)


In [17]:
ADDITIONAL_EPOCHS = 4
LEARNING_RATE = 3e-5

optimizer = torch.optim.AdamW(
    filter(
        lambda parameter: parameter.requires_grad,
        model.parameters()
    ),
    lr=LEARNING_RATE,
    weight_decay=0.01
)

total_steps = len(train_loader) * ADDITIONAL_EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.05 * total_steps),
    num_training_steps=total_steps
)

best_validation_f1 = validation_accuracy * 0
best_model_path = (
    PROJECT_ROOT
    / "models/transformer/best_multilingual_distilbert"
)

patience = 2
epochs_without_improvement = 0

continuation_start = time.perf_counter()

for additional_epoch in range(ADDITIONAL_EPOCHS):
    epoch_number = additional_epoch + 2
    epoch_start = time.perf_counter()

    model.train()
    total_train_loss = 0

    for batch_number, batch in enumerate(train_loader, start=1):
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer.zero_grad()

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()

        if batch_number % 50 == 0:
            print(
                f"Epoch {epoch_number} | "
                f"Batch {batch_number}/{len(train_loader)} | "
                f"Loss: {loss.item():.4f}"
            )

    # Validation
    model.eval()

    validation_labels = []
    validation_predictions = []
    total_validation_loss = 0

    with torch.no_grad():
        for batch in validation_loader:
            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            outputs = model(**batch)
            total_validation_loss += outputs.loss.item()

            predictions = torch.argmax(
                outputs.logits,
                dim=-1
            )

            validation_labels.extend(
                batch["labels"].cpu().numpy()
            )

            validation_predictions.extend(
                predictions.cpu().numpy()
            )

    validation_accuracy = accuracy_score(
        validation_labels,
        validation_predictions
    )

    val_precision, val_recall, val_f1, _ = (
        precision_recall_fscore_support(
            validation_labels,
            validation_predictions,
            average="weighted",
            zero_division=0
        )
    )

    average_train_loss = (
        total_train_loss / len(train_loader)
    )

    average_validation_loss = (
        total_validation_loss / len(validation_loader)
    )

    epoch_minutes = (
        time.perf_counter() - epoch_start
    ) / 60

    epoch_result = {
        "epoch": epoch_number,
        "training_loss": average_train_loss,
        "validation_loss": average_validation_loss,
        "validation_accuracy": validation_accuracy,
        "validation_precision": val_precision,
        "validation_recall": val_recall,
        "validation_f1": val_f1,
        "epoch_minutes": epoch_minutes
    }

    training_history.append(epoch_result)

    print("\nEPOCH RESULT")
    print("-" * 40)
    print(f"Epoch:               {epoch_number}")
    print(f"Training loss:       {average_train_loss:.4f}")
    print(f"Validation loss:     {average_validation_loss:.4f}")
    print(f"Validation accuracy: {validation_accuracy:.4f}")
    print(f"Validation F1:       {val_f1:.4f}")
    print(f"Epoch time:          {epoch_minutes:.2f} minutes")

    if val_f1 > best_validation_f1:
        best_validation_f1 = val_f1
        epochs_without_improvement = 0

        best_model_path.mkdir(
            parents=True,
            exist_ok=True
        )

        model.save_pretrained(best_model_path)
        tokenizer.save_pretrained(best_model_path)

        print("✅ New best model saved!")
    else:
        epochs_without_improvement += 1
        print(
            "No improvement:",
            epochs_without_improvement,
            "epoch(s)"
        )

    if epochs_without_improvement >= patience:
        print("Early stopping activated.")
        break

total_minutes = (
    time.perf_counter() - continuation_start
) / 60

print("\nTraining continuation completed.")
print(f"Best validation F1: {best_validation_f1:.4f}")
print(f"Additional training time: {total_minutes:.2f} minutes")

Epoch 2 | Batch 50/315 | Loss: 0.0000
Epoch 2 | Batch 100/315 | Loss: 0.0000
Epoch 2 | Batch 150/315 | Loss: 0.0000
Epoch 2 | Batch 200/315 | Loss: 0.0000
Epoch 2 | Batch 250/315 | Loss: 0.0000
Epoch 2 | Batch 300/315 | Loss: 0.0000

EPOCH RESULT
----------------------------------------
Epoch:               2
Training loss:       0.0155
Validation loss:     2.1118
Validation accuracy: 0.7194
Validation F1:       0.7260
Epoch time:          4.69 minutes


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ New best model saved!
Epoch 3 | Batch 50/315 | Loss: 0.0000
Epoch 3 | Batch 100/315 | Loss: 0.0000
Epoch 3 | Batch 150/315 | Loss: 0.0000
Epoch 3 | Batch 200/315 | Loss: 0.0000
Epoch 3 | Batch 250/315 | Loss: 0.0000
Epoch 3 | Batch 300/315 | Loss: 0.0000

EPOCH RESULT
----------------------------------------
Epoch:               3
Training loss:       0.0000
Validation loss:     1.7206
Validation accuracy: 0.8000
Validation F1:       0.7894
Epoch time:          4.30 minutes


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ New best model saved!
Epoch 4 | Batch 50/315 | Loss: 0.0000
Epoch 4 | Batch 100/315 | Loss: 0.0000
Epoch 4 | Batch 150/315 | Loss: 0.0000
Epoch 4 | Batch 200/315 | Loss: 0.0000
Epoch 4 | Batch 250/315 | Loss: 0.0000
Epoch 4 | Batch 300/315 | Loss: 0.0000

EPOCH RESULT
----------------------------------------
Epoch:               4
Training loss:       0.0000
Validation loss:     2.0898
Validation accuracy: 0.7806
Validation F1:       0.7611
Epoch time:          4.36 minutes
No improvement: 1 epoch(s)
Epoch 5 | Batch 50/315 | Loss: 0.0000
Epoch 5 | Batch 100/315 | Loss: 0.0000
Epoch 5 | Batch 150/315 | Loss: 0.0000
Epoch 5 | Batch 200/315 | Loss: 0.0000
Epoch 5 | Batch 250/315 | Loss: 0.0000
Epoch 5 | Batch 300/315 | Loss: 0.0000

EPOCH RESULT
----------------------------------------
Epoch:               5
Training loss:       0.0000
Validation loss:     2.0319
Validation accuracy: 0.7972
Validation F1:       0.7791
Epoch time:          4.31 minutes
No improvement: 2 epoch(s)
Early st

In [18]:
from transformers import AutoModelForSequenceClassification

best_model = AutoModelForSequenceClassification.from_pretrained(
    best_model_path
)

best_model.to(device)
best_model.eval()

print("✅ Best Epoch 3 model loaded!")
print("Model path:", best_model_path)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ Best Epoch 3 model loaded!
Model path: C:\Users\delll\Desktop\SmartRoute\models\transformer\best_multilingual_distilbert


In [19]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support
)

test_labels = []
transformer_test_predictions = []
transformer_test_probabilities = []

inference_start = time.perf_counter()

with torch.no_grad():
    for batch in test_loader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        outputs = best_model(**batch)

        probabilities = torch.softmax(
            outputs.logits,
            dim=-1
        )

        predictions = torch.argmax(
            probabilities,
            dim=-1
        )

        test_labels.extend(
            batch["labels"].cpu().numpy()
        )

        transformer_test_predictions.extend(
            predictions.cpu().numpy()
        )

        transformer_test_probabilities.extend(
            probabilities.cpu().numpy()
        )

total_inference_time = (
    time.perf_counter() - inference_start
)

model_latency_ms = (
    total_inference_time / len(test_dataset)
) * 1000

transformer_test_accuracy = accuracy_score(
    test_labels,
    transformer_test_predictions
)

(
    transformer_test_precision,
    transformer_test_recall,
    transformer_test_f1,
    _
) = precision_recall_fscore_support(
    test_labels,
    transformer_test_predictions,
    average="weighted",
    zero_division=0
)

print("TRANSFORMER TEST RESULTS")
print("-" * 40)
print(f"Test accuracy:      {transformer_test_accuracy:.4f}")
print(f"Weighted precision: {transformer_test_precision:.4f}")
print(f"Weighted recall:    {transformer_test_recall:.4f}")
print(f"Weighted F1-score:  {transformer_test_f1:.4f}")
print(f"Model latency:      {model_latency_ms:.4f} ms/complaint")

print("\nClassification report:")
print(
    classification_report(
        test_labels,
        transformer_test_predictions,
        target_names=label_names,
        zero_division=0
    )
)

TRANSFORMER TEST RESULTS
----------------------------------------
Test accuracy:      0.7181
Weighted precision: 0.7874
Weighted recall:    0.7181
Weighted F1-score:  0.7271
Model latency:      20.6441 ms/complaint

Classification report:
                    precision    recall  f1-score   support

     Account Issue       0.33      0.39      0.36        90
Cancellation Issue       0.94      1.00      0.97        90
    Delivery Issue       0.95      0.39      0.55        90
    Fraud/Security       0.35      0.54      0.42        90
     Payment Issue       1.00      0.64      0.78        90
     Product Issue       0.74      0.97      0.84        90
      Refund Issue       1.00      1.00      1.00        90
Return/Replacement       1.00      0.81      0.90        90

          accuracy                           0.72       720
         macro avg       0.79      0.72      0.73       720
      weighted avg       0.79      0.72      0.73       720



In [20]:
import json

transformer_probabilities_array = np.array(
    transformer_test_probabilities
)

transformer_prediction_labels = [
    id2label[int(prediction)]
    for prediction in transformer_test_predictions
]

transformer_confidence = (
    transformer_probabilities_array.max(axis=1)
)

transformer_results = test_df[
    [
        "complaint_id",
        "text",
        "language",
        "category",
        "department",
        "priority"
    ]
].copy()

transformer_results["predicted_category"] = (
    transformer_prediction_labels
)

transformer_results["confidence"] = transformer_confidence

transformer_results["correct"] = (
    transformer_results["category"]
    == transformer_results["predicted_category"]
)

transformer_results["requires_human_review"] = (
    transformer_results["confidence"] < 0.60
)

transformer_predictions_path = (
    PROJECT_ROOT
    / "outputs/reports/transformer_test_predictions.csv"
)

transformer_metrics_path = (
    PROJECT_ROOT
    / "outputs/reports/transformer_metrics.json"
)

transformer_results.to_csv(
    transformer_predictions_path,
    index=False,
    encoding="utf-8-sig"
)

transformer_metrics = {
    "model": "Multilingual DistilBERT",
    "base_model": MODEL_NAME,
    "best_epoch": 3,
    "test_records": int(len(test_df)),
    "accuracy": float(transformer_test_accuracy),
    "weighted_precision": float(transformer_test_precision),
    "weighted_recall": float(transformer_test_recall),
    "weighted_f1": float(transformer_test_f1),
    "model_latency_ms": float(model_latency_ms),
    "human_review_threshold": 0.60,
    "human_review_count": int(
        transformer_results[
            "requires_human_review"
        ].sum()
    )
}

with open(
    transformer_metrics_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        transformer_metrics,
        file,
        indent=4,
        ensure_ascii=False
    )

print("✅ Transformer results saved!")
print(transformer_metrics_path)
print(transformer_predictions_path)

✅ Transformer results saved!
C:\Users\delll\Desktop\SmartRoute\outputs\reports\transformer_metrics.json
C:\Users\delll\Desktop\SmartRoute\outputs\reports\transformer_test_predictions.csv


In [21]:
training_history_df = pd.DataFrame(training_history)

history_path = (
    PROJECT_ROOT
    / "outputs/reports/transformer_training_history.csv"
)

training_history_df.to_csv(
    history_path,
    index=False
)

training_history_df

,epoch,training_loss,validation_loss,validation_accuracy,validation_precision,validation_recall,validation_f1,epoch_minutes
0,1,1.972438,1.828085,0.405556,0.497217,0.405556,0.397999,2.496782
1,2,0.667118,0.587302,0.825000,0.833597,0.825000,0.823652,3.790469
2,3,0.006824,0.562593,0.855556,0.863289,0.855556,0.855858,4.256195
3,4,0.002084,0.561750,0.869444,0.875269,0.869444,0.869013,4.201270
4,5,0.001470,0.567464,0.861111,0.866603,0.861111,0.860989,4.228353
5,2,0.000650,1.485771,0.777778,0.833646,0.777778,0.785477,4.279508
6,3,0.000606,0.628530,0.886111,0.895161,0.886111,0.879063,4.127253
7,4,0.000038,0.512398,0.900000,0.904605,0.900000,0.896869,4.168358
8,5,0.000027,0.517720,0.900000,0.904605,0.900000,0.896869,4.177344
9,2,0.003036,0.493079,0.905556,0.912923,0.905556,0.905156,4.146262
